# test 全流程（ipynb版）

> 目标：在 `/home/viplab/liukejia/project/result` 下分类保存所有结果。

> 流程：
1. 执行 `mv_model/test_grid.ipynb` 推理生图
2. 执行 `reconstruct/single_512_to_glb.ipynb` 重建生图点云
3. 将 `render_3d` 按 2×2 拼接
4. 再执行重建 notebook，重建 `render_3d` 点云

> 输出目录规范：
- `result/inference_images/`：推理图
- `result/reconstruct_pred_glb/`：生图重建点云
- `result/render_3d_stitched/`：拼接后的 render_3d 图
- `result/reconstruct_gt_glb/`：render_3d 重建点云
- `result/executed_notebooks/`：执行后的 notebook（带输出）

In [1]:
from pathlib import Path
import re
import json
import shutil
import tempfile
from typing import Dict, Any, List, Tuple

import cv2
import numpy as np

PROJECT_ROOT = Path('/home/viplab/liukejia/project').resolve()
RESULT_ROOT = PROJECT_ROOT / 'result2'

DIRS = {
    'inference_images': RESULT_ROOT / 'inference_images',
    'reconstruct_pred_glb': RESULT_ROOT / 'reconstruct_pred_glb',
    'render_3d_stitched': RESULT_ROOT / 'render_3d_stitched',
    'reconstruct_gt_glb': RESULT_ROOT / 'reconstruct_gt_glb',
    'executed_notebooks': RESULT_ROOT / 'executed_notebooks',
}

for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print('RESULT_ROOT =', RESULT_ROOT)
for k, v in DIRS.items():
    print(f'- {k}: {v}')

RESULT_ROOT = /home/viplab/liukejia/project/result2
- inference_images: /home/viplab/liukejia/project/result2/inference_images
- reconstruct_pred_glb: /home/viplab/liukejia/project/result2/reconstruct_pred_glb
- render_3d_stitched: /home/viplab/liukejia/project/result2/render_3d_stitched
- reconstruct_gt_glb: /home/viplab/liukejia/project/result2/reconstruct_gt_glb
- executed_notebooks: /home/viplab/liukejia/project/result2/executed_notebooks


In [2]:
# 安装重建流程依赖（若已安装会很快跳过）
import sys, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "-q"])
import gradio
print('gradio version =', gradio.__version__)

gradio version = 6.13.0


In [3]:
# ========== 工具函数 ==========
def _py_literal(v: Any) -> str:
    return repr(v)

def patch_notebook_assignments(src_nb: Path, dst_nb: Path, overrides: Dict[str, Any]) -> Path:
    import nbformat

    nb = nbformat.read(str(src_nb), as_version=4)
    patterns = {
        k: re.compile(rf'^(\\s*){re.escape(k)}\\s*=.*$', flags=re.MULTILINE)
        for k in overrides.keys()
    }
    patched = {k: False for k in overrides.keys()}

    for cell in nb.cells:
        if cell.get('cell_type') != 'code':
            continue
        src = cell.get('source', '')
        for k, v in overrides.items():
            pat = patterns[k]
            src2, n = pat.subn(rf'\\1{k} = {_py_literal(v)}', src)
            if n > 0:
                patched[k] = True
                src = src2
        cell['source'] = src

    # 没命中的变量，追加到第一个 code cell 末尾
    for k, v in overrides.items():
        if not patched[k]:
            for cell in nb.cells:
                if cell.get('cell_type') == 'code':
                    cell['source'] = cell.get('source', '') + f'\\n\\n{k} = {_py_literal(v)}\\n'
                    patched[k] = True
                    break

    dst_nb.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(nb, str(dst_nb))
    return dst_nb

def execute_notebook(nb_path: Path, cwd: Path, timeout: int = -1):
    import nbformat
    from nbconvert.preprocessors import ExecutePreprocessor

    nb = nbformat.read(str(nb_path), as_version=4)
    ep = ExecutePreprocessor(timeout=timeout, kernel_name='python3')
    ep.preprocess(nb, {'metadata': {'path': str(cwd)}})
    nbformat.write(nb, str(nb_path))

def _parse_num_tokens(stem: str):
    parts = stem.split('_')
    if len(parts) < 4:
        return None
    try:
        nums = [int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])]
    except ValueError:
        return None
    return f'{nums[0]}_{nums[1]}_{nums[2]}', nums[3]

def stitch_render3d_to_2x2(render3d_dir: Path, out_dir: Path, target_angles=(0, 90, 180, 270), size=512) -> List[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    files = []
    for ext in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG'):
        files.extend(sorted(render3d_dir.glob(ext)))

    groups: Dict[str, Dict[int, Path]] = {}
    for p in files:
        parsed = _parse_num_tokens(p.stem)
        if parsed is None:
            continue
        num123, a = parsed
        if a in target_angles:
            groups.setdefault(num123, {})[a] = p

    written = []
    idx = 1
    sub = size // 2
    for num123, d in sorted(groups.items(), key=lambda x: tuple(int(t) for t in x[0].split('_'))):
        if not all(a in d for a in target_angles):
            continue
        imgs = []
        ok = True
        for a in target_angles:
            bgr = cv2.imread(str(d[a]), cv2.IMREAD_COLOR)
            if bgr is None:
                ok = False
                break
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb = cv2.resize(rgb, (sub, sub), interpolation=cv2.INTER_AREA)
            imgs.append(rgb)
        if not ok:
            continue
        top = np.concatenate([imgs[0], imgs[1]], axis=1)
        bot = np.concatenate([imgs[2], imgs[3]], axis=1)
        grid = np.concatenate([top, bot], axis=0)
        out_name = f'num123_{num123}_{idx:04d}.png'
        out_path = out_dir / out_name
        cv2.imwrite(str(out_path), cv2.cvtColor(grid, cv2.COLOR_RGB2BGR))
        written.append(out_path)
        idx += 1

    return written

print('工具函数加载完成')

工具函数加载完成


In [4]:
# 修复版2：仅覆写“顶层赋值”，避免误改函数参数/函数体
def patch_notebook_assignments(src_nb: Path, dst_nb: Path, overrides: Dict[str, Any]) -> Path:
    import nbformat

    nb = nbformat.read(str(src_nb), as_version=4)
    keys = list(overrides.keys())
    touched = {k: False for k in keys}

    for cell in nb.cells:
        if cell.get('cell_type') != 'code':
            continue
        lines = cell.get('source', '').splitlines()
        new_lines = []
        for ln in lines:
            replaced = False
            for k in keys:
                # 仅匹配“无缩进”的顶层赋值，如 OUT_DIR = ...
                if re.match(rf'^{re.escape(k)}\s*=.*$', ln):
                    new_lines.append(f"{k} = {_py_literal(overrides[k])}")
                    touched[k] = True
                    replaced = True
                    break
            if not replaced:
                new_lines.append(ln)
        cell['source'] = '\n'.join(new_lines)

    # 若缺失，追加到最后一个 code cell 尾部（顶层执行）
    missing = [k for k in keys if not touched[k]]
    if missing:
        inject = '\n'.join([f"{k} = {_py_literal(overrides[k])}" for k in missing])
        for cell in reversed(nb.cells):
            if cell.get('cell_type') == 'code':
                src = cell.get('source', '')
                cell['source'] = src + '\n\n# injected overrides\n' + inject + '\n'
                break

    dst_nb.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(nb, str(dst_nb))
    return dst_nb

print('patch_notebook_assignments 已切换为修复版2')

patch_notebook_assignments 已切换为修复版2


In [5]:
# # ========== 路径与参数配置（按需改） ==========
# TEST_GRID_NOTEBOOK = PROJECT_ROOT / 'mv_model' / 'test_grid.ipynb'
# RECONSTRUCT_NOTEBOOK = PROJECT_ROOT / 'reconstruct' / 'single_512_to_glb.ipynb'

# # test_grid 输入/权重等若需覆盖可在这里加
# test_grid_overrides = {
#     'OUT_DIR': str(DIRS['inference_images']),
#     # 'NPY_DIR': '/data/test_grid4_from_test_grid4/npy',
#     # 'CAMCONF_DIR': '/data/test_grid4_from_test_grid4/camconf',
#     # 'CKPT_DIR': '/data/mv_model_0_with_drop/checkpoint-21500/controlnet/',
# }

# # 重建：模型输出图 -> glb
# reconstruct_pred_overrides = {
#     'input_path': str(DIRS['inference_images']),
#     'output_path': str(DIRS['reconstruct_pred_glb']),
#     'use_camconf_camera': True,
#     'camconf_dir': '/data/test_grid4_from_test_grid4/camconf',
#     'strict_camconf': True,
#     'post_align_mode': 'camera_fit',
#     'save_npz': False,
# }

# # render_3d 原始目录 -> 先拼接到 result/render_3d_stitched
# RENDER3D_SRC_DIR = Path('/data/test_grid4_from_test_grid4/render_3d')

# # 重建：拼接后的 render_3d 图 -> glb
# reconstruct_gt_overrides = {
#     'input_path': str(DIRS['render_3d_stitched']),
#     'output_path': str(DIRS['reconstruct_gt_glb']),
#     'use_camconf_camera': True,
#     'camconf_dir': '/data/test_grid4_from_test_grid4/camconf',
#     'strict_camconf': True,
#     'post_align_mode': 'camera_fit',
#     'save_npz': False,
# }

# # 执行开关
# RUN_TEST_GRID = True
# RUN_RECONSTRUCT_PRED = True
# RUN_STITCH_RENDER3D = True
# RUN_RECONSTRUCT_GT = True

# print('配置完成')

In [ ]:
# ========== 路径与参数配置（按需改） ==========
TEST_GRID_NOTEBOOK = PROJECT_ROOT / 'mv_model' / 'test_grid.ipynb'
# 改用 vggt 版本（不做你说的那套尺度变换）
RECONSTRUCT_NOTEBOOK = PROJECT_ROOT / 'reconstruct' / 'single_512_to_glb2.ipynb'

# camconf 路径（与数据集保持一致）
CAMCONF_DIR = Path('/data/test_grid5_from_test_grid5/camconf')
NPY_DIR = Path('/data/test_grid5_from_test_grid5/npy')

# test_grid 输入/权重等若需覆盖可在这里加
test_grid_overrides = {
    'OUT_DIR': str(DIRS['inference_images']),
    'NPY_DIR': str(NPY_DIR),
    'CAMCONF_DIR': str(CAMCONF_DIR),
    # 'CKPT_DIR': '/data/mv_model_0_with_drop/checkpoint-21500/controlnet/',
}

# 重建：模型输出图 -> glb（vggt 版本先用最小必要参数）
reconstruct_pred_overrides = {
    'input_path': str(DIRS['inference_images']),
    'output_path': str(DIRS['reconstruct_pred_glb']),
    'save_npz': False,
    'use_camconf_camera': True,
    'camconf_dir': str(CAMCONF_DIR),
    'strict_camconf': True,
}

# render_3d 原始目录 -> 先拼接到 result/render_3d_stitched
RENDER3D_SRC_DIR = Path('/data/test_grid5_from_test_grid5/render_3d')

# 重建：拼接后的 render_3d 图 -> glb
reconstruct_gt_overrides = {
    'input_path': str(DIRS['render_3d_stitched']),
    'output_path': str(DIRS['reconstruct_gt_glb']),
    'save_npz': False,
    # 需要 camera_fit：必须提供对应 camconf
    'camconf_dir': str(CAMCONF_DIR),
    'post_align_mode': 'camera_fit',
    'strict_camconf': True,
}

# 执行开关
RUN_TEST_GRID = True
RUN_RECONSTRUCT_PRED = True
RUN_STITCH_RENDER3D = True
RUN_RECONSTRUCT_GT = True

print('配置完成')
print('RECONSTRUCT_NOTEBOOK =', RECONSTRUCT_NOTEBOOK)

配置完成
RECONSTRUCT_NOTEBOOK = /home/viplab/liukejia/project/reconstruct/single_512_to_glb2.ipynb


In [7]:
# ========== 一键执行全流程 ==========
summary = {}

if RUN_TEST_GRID:
    nb_out = DIRS['executed_notebooks'] / 'test_grid.executed.ipynb'
    patch_notebook_assignments(TEST_GRID_NOTEBOOK, nb_out, test_grid_overrides)
    print('[1/4] 执行 test_grid 推理...')
    execute_notebook(nb_out, cwd=TEST_GRID_NOTEBOOK.parent)
    summary['test_grid_executed'] = str(nb_out)
else:
    print('[1/4] 跳过 test_grid')

if RUN_RECONSTRUCT_PRED:
    nb_out = DIRS['executed_notebooks'] / 'reconstruct_pred.executed.ipynb'
    patch_notebook_assignments(RECONSTRUCT_NOTEBOOK, nb_out, reconstruct_pred_overrides)
    print('[2/4] 执行重建：模型输出图 -> 点云...')
    execute_notebook(nb_out, cwd=RECONSTRUCT_NOTEBOOK.parent)
    summary['reconstruct_pred_executed'] = str(nb_out)
else:
    print('[2/4] 跳过重建(模型输出图)')

if RUN_STITCH_RENDER3D:
    print('[3/4] 拼接 render_3d ...')
    stitched = stitch_render3d_to_2x2(RENDER3D_SRC_DIR, DIRS['render_3d_stitched'])
    print(f'      完成，拼接图数量: {len(stitched)}')
    summary['stitched_render3d_count'] = len(stitched)
    summary['stitched_render3d_dir'] = str(DIRS['render_3d_stitched'])
else:
    print('[3/4] 跳过 render_3d 拼接')

if RUN_RECONSTRUCT_GT:
    nb_out = DIRS['executed_notebooks'] / 'reconstruct_gt.executed.ipynb'
    patch_notebook_assignments(RECONSTRUCT_NOTEBOOK, nb_out, reconstruct_gt_overrides)
    print('[4/4] 执行重建：render_3d 拼接图 -> 点云...')
    execute_notebook(nb_out, cwd=RECONSTRUCT_NOTEBOOK.parent)
    summary['reconstruct_gt_executed'] = str(nb_out)
else:
    print('[4/4] 跳过重建(render_3d)')

summary['result_root'] = str(RESULT_ROOT)
summary['inference_images'] = str(DIRS['inference_images'])
summary['reconstruct_pred_glb'] = str(DIRS['reconstruct_pred_glb'])
summary['render_3d_stitched'] = str(DIRS['render_3d_stitched'])
summary['reconstruct_gt_glb'] = str(DIRS['reconstruct_gt_glb'])
summary['executed_notebooks'] = str(DIRS['executed_notebooks'])

print('\n✅ 全流程完成，结果目录如下：')
print(json.dumps(summary, ensure_ascii=False, indent=2))

[1/4] 执行 test_grid 推理...
[2/4] 执行重建：模型输出图 -> 点云...
[3/4] 拼接 render_3d ...
      完成，拼接图数量: 105
[4/4] 执行重建：render_3d 拼接图 -> 点云...

✅ 全流程完成，结果目录如下：
{
  "test_grid_executed": "/home/viplab/liukejia/project/result2/executed_notebooks/test_grid.executed.ipynb",
  "reconstruct_pred_executed": "/home/viplab/liukejia/project/result2/executed_notebooks/reconstruct_pred.executed.ipynb",
  "stitched_render3d_count": 105,
  "stitched_render3d_dir": "/home/viplab/liukejia/project/result2/render_3d_stitched",
  "reconstruct_gt_executed": "/home/viplab/liukejia/project/result2/executed_notebooks/reconstruct_gt.executed.ipynb",
  "result_root": "/home/viplab/liukejia/project/result2",
  "inference_images": "/home/viplab/liukejia/project/result2/inference_images",
  "reconstruct_pred_glb": "/home/viplab/liukejia/project/result2/reconstruct_pred_glb",
  "render_3d_stitched": "/home/viplab/liukejia/project/result2/render_3d_stitched",
  "reconstruct_gt_glb": "/home/viplab/liukejia/project/result2/reconstr

In [8]:
# ========== 点云评估：CD / F1@τ / Completeness / Accuracy / RMSE ==========
import json
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

try:
    import trimesh
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "trimesh", "-q"])
    import trimesh

METRIC_DIR = RESULT_ROOT / 'metrics'
METRIC_DIR.mkdir(parents=True, exist_ok=True)

PRED_GLB_DIR = DIRS['reconstruct_pred_glb']
GT_GLB_DIR = DIRS['reconstruct_gt_glb']

MAX_POINTS = 120000
F1_THRESHOLDS = [0.2, 0.5]
ALIGN_MODE = 'similarity'  # none | similarity

def load_glb_points(path: Path, max_points=120000, seed=0):
    obj = trimesh.load(str(path), force='scene')
    if isinstance(obj, trimesh.Trimesh):
        pts = np.asarray(obj.vertices, dtype=np.float32)
    else:
        geo = obj.to_geometry()
        pts = np.asarray(geo.vertices, dtype=np.float32)
    if pts.shape[0] == 0:
        raise ValueError(f'空点云: {path}')
    if max_points is not None and pts.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(pts.shape[0], size=max_points, replace=False)
        pts = pts[idx]
    return pts

def nearest_dists(src, dst):
    tree = cKDTree(dst)
    d, _ = tree.query(src, k=1, workers=-1)
    return d.astype(np.float64)

def umeyama_similarity(src, dst):
    src = np.asarray(src, dtype=np.float64)
    dst = np.asarray(dst, dtype=np.float64)
    mu_s, mu_d = src.mean(0), dst.mean(0)
    X, Y = src - mu_s, dst - mu_d
    cov = (Y.T @ X) / src.shape[0]
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U @ Vt) < 0:
        S[-1, -1] = -1
    R = U @ S @ Vt
    var_s = np.mean(np.sum(X * X, axis=1)) + 1e-12
    s = float(np.sum(D * np.diag(S)) / var_s)
    t = mu_d - s * (R @ mu_s)
    return s, R, t

def apply_similarity(pts, s, R, t):
    return (s * (pts @ R.T) + t[None, :]).astype(np.float32)

def compute_metrics(p_rec, p_gt, thresholds=(0.2, 0.5)):
    d_rec = nearest_dists(p_rec, p_gt)
    d_gt = nearest_dists(p_gt, p_rec)

    cd = float(np.mean(d_rec ** 2) + np.mean(d_gt ** 2))
    acc = float(np.mean(d_rec))
    comp = float(np.mean(d_gt))
    rmse = float(np.sqrt((np.mean(d_rec ** 2) + np.mean(d_gt ** 2)) / 2.0))

    out = {
        'cd_l2_squared': cd,
        'accuracy': acc,
        'completeness': comp,
        'rmse_nn': rmse,
    }
    for tau in thresholds:
        p = float(np.mean(d_rec < tau))
        r = float(np.mean(d_gt < tau))
        f1 = 0.0 if (p + r) == 0 else float(2 * p * r / (p + r))
        out[f'precision@{tau}'] = p
        out[f'recall@{tau}'] = r
        out[f'f1@{tau}'] = f1
    return out

pred_files = sorted(PRED_GLB_DIR.glob('*.glb'))
rows = []

for pf in pred_files:
    gf = GT_GLB_DIR / pf.name
    if not gf.exists():
        continue
    try:
        p_rec = load_glb_points(pf, max_points=MAX_POINTS, seed=0)
        p_gt = load_glb_points(gf, max_points=MAX_POINTS, seed=1)

        if ALIGN_MODE == 'similarity':
            n = min(len(p_rec), len(p_gt))
            s, R, t = umeyama_similarity(p_rec[:n], p_gt[:n])
            p_rec_eval = apply_similarity(p_rec, s, R, t)
        else:
            p_rec_eval = p_rec

        m = compute_metrics(p_rec_eval, p_gt, thresholds=F1_THRESHOLDS)
        m['name'] = pf.name
        m['num_rec'] = int(len(p_rec))
        m['num_gt'] = int(len(p_gt))
        rows.append(m)
    except Exception as e:
        rows.append({'name': pf.name, 'error': str(e)})

df = pd.DataFrame(rows)
csv_path = METRIC_DIR / 'pointcloud_metrics.csv'
json_path = METRIC_DIR / 'pointcloud_metrics.json'
df.to_csv(csv_path, index=False, encoding='utf-8')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

ok = df[df.get('error').isna()] if 'error' in df.columns else df
print(f'样本总数: {len(df)}, 成功: {len(ok)}')
if len(ok) > 0:
    cols = [c for c in ['cd_l2_squared', 'accuracy', 'completeness', 'rmse_nn'] + [f'f1@{t}' for t in F1_THRESHOLDS] if c in ok.columns]
    print('平均指标:')
    print(ok[cols].mean(numeric_only=True))

print('CSV ->', csv_path)
print('JSON ->', json_path)
display(df.head(10))

样本总数: 0, 成功: 0
CSV -> /home/viplab/liukejia/project/result2/metrics/pointcloud_metrics.csv
JSON -> /home/viplab/liukejia/project/result2/metrics/pointcloud_metrics.json


""


In [9]:
# ========== 估计“方形”边长（基于重建点云包围盒） ==========
import numpy as np
import pandas as pd

try:
    import trimesh
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'trimesh', '-q'])
    import trimesh

PC_DIR = DIRS['reconstruct_pred_glb']  # 如需看 GT 点云可改为 DIRS['reconstruct_gt_glb']
USE_ROBUST_QUANTILE = True  # True: 用分位数范围更稳健；False: 用min-max
Q_LOW, Q_HIGH = 1.0, 99.0

def load_points(glb_path):
    obj = trimesh.load(str(glb_path), force='scene')
    if isinstance(obj, trimesh.Trimesh):
        pts = np.asarray(obj.vertices, dtype=np.float32)
    else:
        geo = obj.to_geometry()
        pts = np.asarray(geo.vertices, dtype=np.float32)
    return pts

def axis_span(pts, robust=True, q_low=1.0, q_high=99.0):
    if robust:
        lo = np.percentile(pts, q_low, axis=0)
        hi = np.percentile(pts, q_high, axis=0)
    else:
        lo = pts.min(axis=0)
        hi = pts.max(axis=0)
    return (hi - lo).astype(np.float64)  # [dx, dy, dz]

rows = []
for glb in sorted(PC_DIR.glob('*.glb')):
    pts = load_points(glb)
    if pts.shape[0] == 0:
        continue
    dx, dy, dz = axis_span(pts, robust=USE_ROBUST_QUANTILE, q_low=Q_LOW, q_high=Q_HIGH)
    side = (dx + dy) / 2.0  # 方形边长近似：XY 两边均值
    squareness = abs(dx - dy) / (side + 1e-12)
    rows.append({
        'name': glb.name,
        'n_points': int(pts.shape[0]),
        'dx': float(dx),
        'dy': float(dy),
        'dz(height)': float(dz),
        'side_est': float(side),
        'squareness_rel_diff': float(squareness),
    })

df_box = pd.DataFrame(rows)
if len(df_box) == 0:
    print('未找到可用 glb 点云。')
else:
    print(f'样本数: {len(df_box)}')
    print('边长估计 side_est 的统计：')
    print(df_box['side_est'].describe())
    print('\n平均高度 dz(height):', df_box['dz(height)'].mean())
    print('平均方形程度 |dx-dy|/side:', df_box['squareness_rel_diff'].mean())
    display(df_box.head(10))

    out_csv = RESULT_ROOT / 'metrics' / 'box_size_estimation.csv'
    df_box.to_csv(out_csv, index=False, encoding='utf-8')
    print('已保存:', out_csv)

样本数: 9
边长估计 side_est 的统计：
count      9.000000
mean     199.359497
std        0.018289
min      199.330727
25%      199.345451
50%      199.366364
75%      199.367905
max      199.388901
Name: side_est, dtype: float64

平均高度 dz(height): 24.429417928059895
平均方形程度 |dx-dy|/side: 2.7213313541485697e-07


,name,n_points,dx,dy,dz(height),side_est,squareness_rel_diff
0,num123_-800_100_0_0007.glb,56,199.367859,199.367950,24.106461,199.367905,4.592150e-07
1,num123_-800_150_0_0008.glb,56,199.358948,199.359009,24.448700,199.358978,3.061570e-07
2,num123_-800_200_0_0009.glb,56,199.388855,199.388947,23.307129,199.388901,4.591667e-07
3,num123_-850_100_0_0004.glb,56,199.345459,199.345444,24.964996,199.345451,7.654446e-08
4,num123_-850_150_0_0005.glb,56,199.338257,199.338211,25.242050,199.338234,2.296417e-07
5,num123_-850_200_0_0006.glb,56,199.372437,199.372406,23.937485,199.372421,1.530682e-07
6,num123_-900_100_0_0001.glb,56,199.366516,199.366470,24.162598,199.366493,2.296091e-07
7,num123_-900_150_0_0002.glb,56,199.330750,199.330704,25.527161,199.330727,2.296503e-07
8,num123_-900_200_0_0003.glb,56,199.366394,199.366333,24.168182,199.366364,3.061457e-07


已保存: /home/viplab/liukejia/project/result2/metrics/box_size_estimation.csv


In [10]:
# === 方形边长估计：分别统计 GT 与 Pred 每个点云 ===
import numpy as np
import pandas as pd
import trimesh
from pathlib import Path

Q_LOW = 0.01
Q_HIGH = 0.99

# 兼容不同版本变量命名
if 'METRIC_DIR' in globals():
    _metric_dir = Path(METRIC_DIR)
elif 'DIRS' in globals() and isinstance(DIRS, dict) and 'metrics' in DIRS:
    _metric_dir = Path(DIRS['metrics'])
else:
    _metric_dir = Path(PROJECT_ROOT) / 'result' / 'metrics'
_metric_dir.mkdir(parents=True, exist_ok=True)

def estimate_square_side_for_dir(glb_dir: Path, split_name: str) -> pd.DataFrame:
    rows = []
    files = sorted(glb_dir.glob('*.glb'))
    for gf in files:
        try:
            m = trimesh.load(gf, force='mesh')
            pts = np.asarray(m.vertices)
            if pts.ndim != 2 or pts.shape[0] < 32:
                continue

            lo = np.quantile(pts, Q_LOW, axis=0)
            hi = np.quantile(pts, Q_HIGH, axis=0)
            dx, dy, dz = (hi - lo).astype(float)
            side = float((dx + dy) / 2.0)
            squareness_rel = float(abs(dx - dy) / max(side, 1e-12))

            rows.append({
                'split': split_name,
                'name': gf.stem,
                'dx': dx,
                'dy': dy,
                'dz': dz,
                'side_est': side,
                'squareness_rel_diff': squareness_rel,
            })
        except Exception:
            continue

    return pd.DataFrame(rows)

pred_df = estimate_square_side_for_dir(Path(DIRS['reconstruct_pred_glb']), 'pred')
gt_df = estimate_square_side_for_dir(Path(DIRS['reconstruct_gt_glb']), 'gt')
all_df = pd.concat([pred_df, gt_df], ignore_index=True)

out_csv = _metric_dir / 'box_size_estimation_pred_gt.csv'
all_df.to_csv(out_csv, index=False)

print('保存：', out_csv)

for split_name, cur in [('pred', pred_df), ('gt', gt_df)]:
    if len(cur) == 0:
        print(f'[{split_name}] 无有效样本')
        continue
    print('\n' + '=' * 60)
    print(f'[{split_name}] 样本数: {len(cur)}')
    print(cur[['side_est', 'dz', 'squareness_rel_diff']].describe().loc[['mean','50%','min','max']])

# 可配对比较（同名 stem）
if len(pred_df) and len(gt_df):
    merged = pred_df[['name','side_est']].merge(
        gt_df[['name','side_est']], on='name', suffixes=('_pred','_gt')
    )
    if len(merged):
        merged['abs_diff'] = (merged['side_est_pred'] - merged['side_est_gt']).abs()
        merged['rel_diff'] = merged['abs_diff'] / np.maximum(merged['side_est_gt'].abs(), 1e-12)
        print('\n' + '=' * 60)
        print(f'[配对样本] 数量: {len(merged)}')
        print('平均绝对差(abs_diff):', float(merged['abs_diff'].mean()))
        print('中位绝对差(abs_diff):', float(merged['abs_diff'].median()))
        print('平均相对差(rel_diff):', float(merged['rel_diff'].mean()))
        pair_csv = _metric_dir / 'box_size_estimation_pred_gt_paired.csv'
        merged.to_csv(pair_csv, index=False)
        print('保存：', pair_csv)


保存： /home/viplab/liukejia/project/result2/metrics/box_size_estimation_pred_gt.csv

[pred] 样本数: 9
        side_est         dz  squareness_rel_diff
mean  199.359491  24.429422         2.759655e-07
50%   199.366357  24.168188         2.104203e-07
min   199.330726  23.307143         8.133144e-08
max   199.388899  25.527159         5.643923e-07

[gt] 样本数: 105
        side_est         dz  squareness_rel_diff
mean  199.382195  23.563417         3.121625e-07
50%   199.378992  23.687071         2.181604e-07
min   199.337521  13.890028         0.000000e+00
max   199.635839  25.268250         1.202520e-06
